# Camera-pilot weight audit — CPU only, no training
**PARKING-CAMERA-WEIGHT-AUDIT-2026-10-27-r1**

The five-epoch pilot completed, but both arms selected epoch 0. This notebook collects the already saved epoch-3 checkpoints and compares existing depth weights, depth BatchNorm statistics and new adapter weights. It does not use images, stage a dataset, run inference, allocate a GPU, restore the old runtime, train, export or update the phone.

Use a CPU runtime. Run only the four sections below, not the earlier training notebook. Tensor movement alone cannot prove why accuracy regressed; return the final ZIP for a subsequent fixed-data counterfactual analysis. The completed training folder is read-only.

## 1. Mount Drive and update main

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import collections, datetime, os, shlex, subprocess, sys
REVISION = 'PARKING-CAMERA-WEIGHT-AUDIT-2026-10-27-r1'
MOBILE = Path('/content/mobile_adas3d')
BASE = Path('/content/drive/MyDrive/mobile_adas3d_outputs')
RUN = BASE/'parking_shared_camera/shared_camera_5epoch_r1'
OUTPUT = BASE/'parking_shared_camera_audit/shared_camera_epoch3_weight_audit_r1'
SCRIPT = MOBILE/'scripts/audit_parking_shared_camera_weights.py'
INPUTS = OUTPUT/'parking_shared_camera_epoch3_inputs.zip'
OUTPUT.mkdir(parents=True, exist_ok=True)
def run(command, name, cwd=None):
    command = [str(v) for v in command]
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    log = OUTPUT/'colab_logs'/f'{name}_{stamp}.log'
    log.parent.mkdir(parents=True, exist_ok=True)
    print('+', shlex.join(command), '\nDurable log:', log, flush=True)
    env = os.environ.copy(); env['PYTHONPATH'] = str(MOBILE)
    tail = collections.deque(maxlen=25)
    with log.open('w') as stream:
        p = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in p.stdout:
                print(line, end='', flush=True); stream.write(line); stream.flush(); tail.append(line.rstrip())
        finally:
            p.stdout.close()
        code = p.wait(); stream.write(f'\nExit code: {code}\n')
    if code: raise RuntimeError(f'Exit {code}; inspect {log}\n'+'\n'.join(tail))
url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE.exists(): run(['git','clone',url,MOBILE], 'clone')
branch = subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE,text=True).strip()
if branch != 'main': raise RuntimeError(f'Use main, not {branch!r}; preserve existing branches')
run(['git','pull','--ff-only'], 'update_main', MOBILE)
if not SCRIPT.is_file(): raise FileNotFoundError(SCRIPT)
print(REVISION, '\nRead-only completed run:', RUN, '\nSeparate audit output:', OUTPUT)


## 2. Collect the existing epoch-3 files
This uses only Python's standard library. If a checkpoint is missing, stop and preserve the run—do not restart training.

In [ ]:
run([sys.executable,'-u',SCRIPT,'collect','--run',RUN,'--output',OUTPUT], 'collect', MOBILE)
assert INPUTS.is_file()


## 3. Run tests and compare tensors on CPU
Uses the Colab runtime's existing PyTorch, with safe weights-only loading and map_location=cpu. No package installation or training-runtime identity match is needed for this tensor-only audit.

In [ ]:
run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_parking_shared_camera_weights.py','-v'], 'audit_tests', MOBILE)
run([sys.executable,'-u',SCRIPT,'audit','--inputs',INPUTS,'--output',OUTPUT], 'weight_audit', MOBILE)
import json
report = json.loads((OUTPUT/'camera_weight_audit.json').read_text())
assert report['complete'] and report['optimizer_steps']==0
assert not report['forward_inference_performed'] and not report['training_performed']
print(json.dumps({arm: {key: report['arms'][arm][key] for key in ['adapter_weights_moved','existing_depth_weights_moved','existing_depth_bn_statistics_moved']} for arm in report['arms']}, indent=2))


## 4. Download and return the audit ZIP
Includes the report and both trained epoch-3 checkpoints so the next analysis can be done without another training run.

In [ ]:
run([sys.executable,'-u',SCRIPT,'bundle','--inputs',INPUTS,'--output',OUTPUT], 'bundle_audit', MOBILE)
BUNDLE = OUTPUT/'parking_shared_camera_weight_audit_results.zip'
print('Return this ZIP:', BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
